[https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29](https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29)

In [21]:
from datasets import load_dataset

In [22]:
datapath = "/mnt/d/Data/CS/workspace/project/数据格式转换"
data_files = {"train": datapath + "/drugsComTrain_raw.tsv", "test": datapath + "/drugsComTest_raw.tsv"}
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")

In [23]:
drug_sample = drug_dataset["train"].shuffle(seed=42).select(range(1000))
drug_sample[:5]

{'Unnamed: 0': [87571, 178045, 80482, 159268, 205477],
 'drugName': ['Naproxen', 'Duloxetine', 'Mobic', 'TriNessa', 'Pristiq'],
 'condition': ['Gout, Acute',
  'ibromyalgia',
  'Inflammatory Conditions',
  'Birth Control',
  'Depression'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."',

In [24]:
drug_sample

Dataset({
    features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
    num_rows: 1000
})

更改列名

In [25]:
for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len(drug_dataset[split].unique("Unnamed: 0"))

In [26]:
drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)

In [27]:
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

In [28]:
def lowercase_condition(example):
    return {"condition": example["condition"].lower()}
drug_dataset = drug_dataset.map(lowercase_condition)

Map:   0%|          | 30/161297 [00:00<00:27, 5910.80 examples/s]


AttributeError: 'NoneType' object has no attribute 'lower'

过滤空数据

In [29]:
drug_dataset = drug_dataset.filter(lambda example: example["condition"] is not None)
drug_dataset

Filter: 100%|██████████| 53766/53766 [00:00<00:00, 330270.90 examples/s]


DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 160398
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53471
    })
})

In [30]:
drug_dataset = drug_dataset.map(lowercase_condition)

Map: 100%|██████████| 53471/53471 [00:02<00:00, 21748.02 examples/s]


In [ ]:
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 160398
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53471
    })
})

统计每一条评论的长度

In [31]:
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}
drug_dataset = drug_dataset.map(compute_review_length)
drug_dataset["train"][0]

Map: 100%|██████████| 53471/53471 [00:02<00:00, 26600.00 examples/s]


{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'left ventricular dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

In [32]:
drug_dataset["train"].sort("review_length")[:3]

{'patient_id': [111469, 13653, 53602],
 'drugName': ['Ledipasvir / sofosbuvir',
  'Amphetamine / dextroamphetamine',
  'Alesse'],
 'condition': ['hepatitis c', 'adhd', 'birth control'],
 'review': ['"Headache"', '"Great"', '"Awesome"'],
 'rating': [10.0, 10.0, 10.0],
 'date': ['February 3, 2015', 'October 20, 2009', 'November 23, 2015'],
 'usefulCount': [41, 3, 0],
 'review_length': [1, 1, 1]}

In [33]:
drug_dataset = drug_dataset.filter(lambda example: example["review_length"] >= 30)
drug_dataset

Filter: 100%|██████████| 53471/53471 [00:00<00:00, 292565.04 examples/s]


DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 139529
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46434
    })
})

In [34]:
drug_dataset["train"][0]

{'patient_id': 95260,
 'drugName': 'Guanfacine',
 'condition': 'adhd',
 'review': '"My son is halfway through his fourth week of Intuniv. We became concerned when he began this last week, when he started taking the highest dose he will be on. For two days, he could hardly get out of bed, was very cranky, and slept for nearly 8 hours on a drive home from school vacation (very unusual for him.) I called his doctor on Monday morning and she said to stick it out a few days. See how he did at school, and with getting up in the morning. The last two days have been problem free. He is MUCH more agreeable than ever. He is less emotional (a good thing), less cranky. He is remembering all the things he should. Overall his behavior is better. \r\nWe have tried many different medications and so far this is the most effective."',
 'rating': 8.0,
 'date': 'April 27, 2010',
 'usefulCount': 192,
 'review_length': 141}

In [35]:
import html
text = "I&#39;m a transformer called BERT"
html.unescape(text)

"I'm a transformer called BERT"

In [36]:
drug_dataset = drug_dataset.map(lambda x: {"review": html.unescape(x["review"])})

Map: 100%|██████████| 46434/46434 [00:02<00:00, 17519.97 examples/s]


In [ ]:
new_drug_dataset = drug_dataset.map(
    lambda x: {"review": html.unescape(o) for o in x["review"]}, batched=True
)

In [39]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [40]:
def tokenize_function(example):
    return tokenizer(example["review"], truncation=True, max_length=128, return_overflowing_tokens=True)

In [41]:
drug_dataset["train"][0]

{'patient_id': 95260,
 'drugName': 'Guanfacine',
 'condition': 'adhd',
 'review': '"My son is halfway through his fourth week of Intuniv. We became concerned when he began this last week, when he started taking the highest dose he will be on. For two days, he could hardly get out of bed, was very cranky, and slept for nearly 8 hours on a drive home from school vacation (very unusual for him.) I called his doctor on Monday morning and she said to stick it out a few days. See how he did at school, and with getting up in the morning. The last two days have been problem free. He is MUCH more agreeable than ever. He is less emotional (a good thing), less cranky. He is remembering all the things he should. Overall his behavior is better. \r\nWe have tried many different medications and so far this is the most effective."',
 'rating': 8.0,
 'date': 'April 27, 2010',
 'usefulCount': 192,
 'review_length': 141}

In [42]:
result = tokenize_function(drug_dataset["train"][0])
[len(inp) for inp in result["input_ids"]]

[128, 45]

In [43]:
result

{'input_ids': [[101, 1000, 2026, 2365, 2003, 8576, 2083, 2010, 2959, 2733, 1997, 20014, 19496, 2615, 1012, 2057, 2150, 4986, 2043, 2002, 2211, 2023, 2197, 2733, 1010, 2043, 2002, 2318, 2635, 1996, 3284, 13004, 2002, 2097, 2022, 2006, 1012, 2005, 2048, 2420, 1010, 2002, 2071, 6684, 2131, 2041, 1997, 2793, 1010, 2001, 2200, 27987, 2100, 1010, 1998, 7771, 2005, 3053, 1022, 2847, 2006, 1037, 3298, 2188, 2013, 2082, 10885, 1006, 2200, 5866, 2005, 2032, 1012, 1007, 1045, 2170, 2010, 3460, 2006, 6928, 2851, 1998, 2016, 2056, 2000, 6293, 2009, 2041, 1037, 2261, 2420, 1012, 2156, 2129, 2002, 2106, 2012, 2082, 1010, 1998, 2007, 2893, 2039, 1999, 1996, 2851, 1012, 1996, 2197, 2048, 2420, 2031, 2042, 3291, 2489, 1012, 2002, 2003, 2172, 2062, 5993, 3085, 2084, 2412, 1012, 2002, 2003, 102], [101, 2625, 6832, 1006, 1037, 2204, 2518, 1007, 1010, 2625, 27987, 2100, 1012, 2002, 2003, 10397, 2035, 1996, 2477, 2002, 2323, 1012, 3452, 2010, 5248, 2003, 2488, 1012, 2057, 2031, 2699, 2116, 2367, 20992, 1998,

In [45]:
drug_dataset["train"].column_names

['patient_id',
 'drugName',
 'condition',
 'review',
 'rating',
 'date',
 'usefulCount',
 'review_length']

In [46]:
tokenized_dataset = drug_dataset.map(
    tokenize_function, batched=True, remove_columns=drug_dataset["train"].column_names
)

Map: 100%|██████████| 46434/46434 [00:02<00:00, 21217.47 examples/s]


In [47]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'overflow_to_sample_mapping'],
        num_rows: 205213
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'overflow_to_sample_mapping'],
        num_rows: 68349
    })
})

In [48]:
drug_dataset.set_format("pandas")
drug_dataset["train"][:3]

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,95260,Guanfacine,adhd,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,141
1,92703,Lybrel,birth control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,134
2,138000,Ortho Evra,birth control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10,89


In [50]:
train_df = drug_dataset["train"][:]

In [51]:
frequencies = (
    train_df["condition"]
    .value_counts()
    .to_frame()
    .reset_index()
    .rename(columns={"index": "condition", "condition": "frequency"})
)
frequencies.head()

,frequency,count
0,birth control,27740
1,depression,8057
2,acne,5225
3,anxiety,5033
4,pain,4788


In [53]:
from datasets import Dataset
freq_datqset = Dataset.from_pandas(frequencies)
freq_datqset

Dataset({
    features: ['frequency', 'count'],
    num_rows: 821
})

切分数据集

In [57]:
drug_dataset.reset_format()
drug_dataset_clean = drug_dataset["train"].train_test_split(train_size=0.8, seed=42)
drug_dataset_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 111623
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27906
    })
})

In [58]:
drug_dataset_clean["validation"] = drug_dataset_clean.pop("test")
drug_dataset_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 111623
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27906
    })
})

In [59]:
drug_dataset_clean.save_to_disk(datapath+"/drug_dataset_clean")

Saving the dataset (1/1 shards): 100%|██████████| 27906/27906 [00:00<00:00, 98507.77 examples/s] 


In [60]:
from datasets import load_from_disk
drug_dataset_reloaded = load_from_disk(datapath+"/drug_dataset_clean")
drug_dataset_reloaded

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 111623
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27906
    })
})